# Boltz-2 · TREM2 ± Aβ42 결합 예측

**무엇을 왜 하는가**

논문(Yao et al. 2024)은 AlphaFold3로 변이의 **구조** 변화를 보여줬다.
우리가 재현해보니 그 변화는 모델 자체의 노이즈와 구분되지 않았다.

그런데 실험 문헌은 TREM2 변이가 **Aβ 결합**을 떨어뜨린다고 보고한다.
→ 봐야 할 건 구조가 아니라 **결합**이다.
→ AlphaFold3는 결합을 못 잰다. **Boltz-2는 잰다.**

| 근거 | 내용 |
|---|---|
| Zhao et al. 2018, *Neuron* | TREM2가 Aβ 올리고머에 나노몰 친화도 결합. AD 변이가 결합 감소 |
| Zhong et al. 2018, *Mol Neurodegener* | oAβ1-42 고친화도 결합. 결합 필수 잔기 **31–91** (R62 포함) |
| Yeh et al. 2016, *Neuron* | TREM2–APOE/CLU/LDL 결합, 질병 변이가 저해 |
| Passaro et al. 2025 | Boltz-2 (MIT 라이선스, FEP 근접 성능) |

**⚠️ 한계 — 반드시 함께 보고할 것**

1. Boltz-2의 affinity head는 **저분자 전용**이다. Aβ42는 펩타이드라 결합력 수치가 안 나온다.
   우리가 얻는 건 **ipTM(인터페이스 신뢰도, 0~1)** 이다.
2. Bret et al. 2026 (*JCIM*)은 Boltz-2가 **결합부위 변이에 둔감**하다고 보고했다.
   → ipTM 차이도 **노이즈 바닥 검정**을 거쳐야 한다. 그냥 믿으면 안 된다.

---

## 실행 순서

1. **런타임 → 런타임 유형 변경 → T4 GPU** 로 먼저 바꾼다
2. 셀을 위에서부터 순서대로 실행
3. 2번 셀(설치) 후 **런타임 재시작**이 뜨면 재시작하고 3번 셀부터 이어서
4. 마지막 셀에서 결과 ZIP이 다운로드된다

## 1. GPU 확인

`Tesla T4` 같은 이름이 나와야 한다. 안 나오면 런타임 유형을 GPU로 바꿀 것.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Boltz-2 설치

2~3분 걸린다. 끝나고 "런타임을 다시 시작" 버튼이 뜨면 **누르고 3번 셀부터** 이어서 실행.

In [ ]:
import sys, subprocess, shutil

print("Python:", sys.version.split()[0])
print("boltz 는 Python 3.10~3.12 만 지원한다 (>=3.10,<3.13)\n")

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:]); print("STDERR:", r.stderr[-1500:])
    return r.returncode == 0

BOLTZ = "boltz"   # 아래에서 필요하면 바뀐다

if sys.version_info[:2] < (3, 13):
    # 정상 경로 — 시스템 파이썬에 그대로 설치
    ok = sh(f"{sys.executable} -m pip install 'boltz[cuda]==2.2.1'")
else:
    # Colab 파이썬이 3.13 이상이면 boltz 를 못 넣는다 → 3.11 가상환경을 따로 만든다
    print("Python 3.13+ 감지 → 3.11 환경을 따로 만든다 (condacolab 보다 빠름)")
    sh(f"{sys.executable} -m pip install -q uv")
    sh("uv venv --python 3.11 /content/bz")
    ok = sh("uv pip install --python /content/bz/bin/python 'boltz[cuda]==2.2.1'")
    BOLTZ = "/content/bz/bin/boltz"

# 설치 확인 — 여기서 걸러야 나중 셀에서 command not found 가 안 난다
found = shutil.which(BOLTZ) or (BOLTZ if BOLTZ.startswith("/") and __import__("os").path.exists(BOLTZ) else None)
print("\nboltz 실행파일:", found or "못 찾음")
assert found, "boltz 설치 실패 — 위 STDERR 를 확인할 것"

# 다음 셀들이 쓸 수 있게 파일로 남긴다 (런타임 재시작해도 유지)
open("/content/BOLTZ_BIN.txt", "w").write(BOLTZ)
print("설치 완료. 런타임 재시작 안내가 뜨면 재시작 후 3번 셀부터 실행")


## 3. 입력 서열 작성 + 검증

서열을 코드 안에 박아 두었으므로 업로드할 파일이 없다.
`assert`로 잔기 번호가 맞는지 강제 확인한다 — 깨지면 그 자리에서 멈춘다.

In [ ]:
from pathlib import Path

# ── TREM2 Ig 도메인 19–130 ────────────────────────────────────────────
# UniProt Q9NZC2 의 19–130 구간. 결정구조 5UD8·5ELI 가 덮는 범위이고,
# Aβ 결합에 필수인 31–91 잔기를 모두 포함한다 (Zhong et al. 2018).
# 전장을 쓰면 막관통·세포질 꼬리가 인터페이스 지표를 오염시키므로 도메인만 쓴다.
TREM2_WT = (
    "HNTTVFQGVAGQSLQVSCPYDSMKHWGRRKAWCRQLGEKGPCQRVVSTHNLWLLSFLRRWNGSTAITDDT"
    "LGGTLTITLRNLQPHDAGLYQCQSLHGSEADTLRKVLVEVLA"
)
MUT_POS_IN_DOMAIN = 62 - 19 + 1   # 전구체 62번 → 도메인 안에서 44번째

assert len(TREM2_WT) == 112, f"TREM2 도메인 길이 {len(TREM2_WT)} — 112를 기대"
assert TREM2_WT[MUT_POS_IN_DOMAIN - 1] == "R", "62번이 Arg가 아니다 — 서열 확인 필요"

TREM2_R62H = (
    TREM2_WT[: MUT_POS_IN_DOMAIN - 1] + "H" + TREM2_WT[MUT_POS_IN_DOMAIN :]
)
assert sum(a != b for a, b in zip(TREM2_WT, TREM2_R62H)) == 1, "변이가 1개가 아니다"

# ── Aβ42 = APP(P05067) 672–713 ───────────────────────────────────────
AB42 = "DAEFRHDSGYEVHHQKLVFFAEDVGSNKGAIIGLMVGGVVIA"
assert len(AB42) == 42, "Aβ42 길이 오류"

print(f"TREM2 WT    {len(TREM2_WT)} aa   62번 = {TREM2_WT[MUT_POS_IN_DOMAIN-1]}")
print(f"TREM2 R62H  {len(TREM2_R62H)} aa   62번 = {TREM2_R62H[MUT_POS_IN_DOMAIN-1]}")
print(f"Ab42        {len(AB42)} aa")
print("검증 통과\n")

# ── YAML 작성 ─────────────────────────────────────────────────────────
# ⚠️ properties: affinity 블록을 일부러 넣지 않는다.
#    Boltz-2 affinity head는 저분자(ligand) 전용이고 Ab42는 펩타이드다.
Path("inputs").mkdir(exist_ok=True)

def write_yaml(name, trem2, with_peptide=True):
    y = ["version: 1", "sequences:", "  - protein:", "      id: A",
         f'      sequence: "{trem2}"']
    if with_peptide:
        y += ["  - protein:", "      id: B", f'      sequence: "{AB42}"']
    p = Path("inputs") / f"{name}.yaml"
    p.write_text("\n".join(y) + "\n")
    return p

JOBS = [
    ("TREM2_WT_AB42",   TREM2_WT,   True),
    ("TREM2_R62H_AB42", TREM2_R62H, True),
]
for name, seq, wp in JOBS:
    print("작성:", write_yaml(name, seq, wp))

## 4. 사용 가능한 옵션 확인

버전에 따라 플래그 이름이 다를 수 있어 먼저 확인한다.
특히 `--diffusion_samples`(O) / `--num_diffusion_samples`(X) 를 헷갈리기 쉽다.

In [ ]:
BOLTZ = open("/content/BOLTZ_BIN.txt").read().strip()
print("사용할 실행파일:", BOLTZ)
!{BOLTZ} predict --help 2>&1 | head -60


## 5. 예측 실행

복합체 2개(WT / R62H) × 시드 3개 = **6회**.
각 회마다 모델 5개가 나오므로 총 **구조 30개** — 구조 파트(AlphaFold3)와 같은 규모다.

T4 기준 1회 3~8분, 전체 **30분~1시간** 예상. MSA 서버 대기가 포함된다.

> 이미 끝난 폴더는 건너뛰므로, 중간에 끊겨도 이 셀을 다시 실행하면 이어서 돈다.

In [ ]:
BOLTZ = open("/content/BOLTZ_BIN.txt").read().strip()

import subprocess, time
from pathlib import Path

SEEDS = [1, 2, 3]
DIFFUSION_SAMPLES = 5

Path("outputs").mkdir(exist_ok=True)

for name, _, _ in JOBS:
    for seed in SEEDS:
        out = Path("outputs") / f"{name}_seed{seed}"
        if out.exists() and any(out.rglob("*.cif")):
            print(f"건너뜀 (이미 있음): {out.name}")
            continue

        cmd = [
            BOLTZ, "predict", f"inputs/{name}.yaml",
            "--out_dir", str(out),
            "--use_msa_server",
            "--diffusion_samples", str(DIFFUSION_SAMPLES),
            "--seed", str(seed),
            "--output_format", "mmcif",
        ]
        print(f"\n▶ {name}  seed {seed}")
        t0 = time.time()
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            # T4(sm_75)에서 cuequivariance 커널 오류가 나면 --no_kernels 로 폴백
            print("  1차 실패 → --no_kernels 로 재시도")
            r = subprocess.run(cmd + ["--no_kernels"], capture_output=True, text=True)
        if r.returncode != 0:
            print("  실패:\n", r.stderr[-1500:])
        else:
            print(f"  완료 ({time.time()-t0:.0f}초)")

## 6. ipTM 집계

**ipTM이 결합 지표다.** 두 사슬(TREM2 · Aβ42) 사이 인터페이스를 모델이 얼마나 확신하는지.

WT와 R62H를 비교하되, **차이가 노이즈보다 큰지**를 함께 본다.

In [ ]:
import json, itertools, statistics as st
from pathlib import Path
import pandas as pd
from scipy.stats import mannwhitneyu

rows = []
for conf in sorted(Path("outputs").rglob("confidence_*.json")):
    d = json.loads(conf.read_text())
    parts = conf.parents[3].name.split("_seed")   # TREM2_WT_AB42_seed1
    job, seed = parts[0], parts[1] if len(parts) > 1 else "?"
    rows.append({
        "job": job,
        "allele": "WT" if "_WT_" in job else "R62H",
        "seed": seed,
        "model": conf.stem.rsplit("_", 1)[-1],
        "iptm": d.get("iptm"),
        "ptm": d.get("ptm"),
        "complex_plddt": d.get("complex_plddt"),
        "confidence_score": d.get("confidence_score"),
    })

df = pd.DataFrame(rows)
if df.empty:
    print("결과 없음 — 5번 셀이 성공했는지 확인")
else:
    df.to_csv("boltz_iptm.csv", index=False)
    display(df.groupby("allele")[["iptm", "ptm", "complex_plddt"]]
              .agg(["count", "median", "min", "max"]).round(3))

    wt  = df[df.allele == "WT"]["iptm"].dropna().tolist()
    mut = df[df.allele == "R62H"]["iptm"].dropna().tolist()

    # 노이즈: 같은 대립끼리의 모델 간 ipTM 차이
    noise = [abs(a - b) for grp in (wt, mut)
                        for a, b in itertools.combinations(grp, 2)]
    # 신호: WT × R62H 전조합의 ipTM 차이
    signal = [abs(a - b) for a in wt for b in mut]

    if noise and signal:
        u, p = mannwhitneyu(signal, noise, alternative="greater")
        print(f"\nipTM  WT   중앙값 {st.median(wt):.3f}  (n={len(wt)})")
        print(f"ipTM  R62H 중앙값 {st.median(mut):.3f}  (n={len(mut)})")
        print(f"\n노이즈(같은 대립 모델 간 차이) 중앙값 {st.median(noise):.4f}  n={len(noise)}")
        print(f"신호  (WT vs R62H 차이)        중앙값 {st.median(signal):.4f}  n={len(signal)}")
        print(f"Mann-Whitney U 단측 p = {p:.3g}")
        print("\n판정:", "신호 > 노이즈 — 결합 차이 지지" if p < 0.05
              else "구분 안 됨 — Boltz-2가 이 변이를 보지 못함")
        print("\n※ Bret et al. 2026은 Boltz-2가 결합부위 변이에 둔감하다고 보고했다.")
        print("   '구분 안 됨'이 나와도 그것 자체가 보고할 만한 결과다.")

## 7. 결과 내려받기

ZIP을 받아 프로젝트 폴더에 풀면 된다.

In [ ]:
!zip -qr boltz_trem2_ab42.zip outputs inputs boltz_iptm.csv 2>/dev/null
!ls -lh boltz_trem2_ab42.zip

from google.colab import files
files.download("boltz_trem2_ab42.zip")